# Pre-work - before Day 1

> Twenty minutes. Confirms your environment works and refreshes the pandas and regression bits the course leans on.

Twenty minutes, before Day 1. Two jobs:

1. **Prove your environment works** - so Day 1 opens with content, not an install clinic.
2. **Refresh the pandas and regression bits** the course leans on.

Nothing here is assessed. If a section is already obvious to you, skim it.

---
## 1. Does the environment work?

From the repo root, in a terminal:

```bash
uv sync --extra dev
.venv/Scripts/python.exe scripts/check_env.py
.venv/Scripts/python.exe scripts/prefetch_data.py
```

`check_env.py` must print **"Ready. See you on Day 1."** If it does not, it tells you
exactly what to fix. `prefetch_data.py` needs internet **once**; after that the course
runs offline.

Then run the cell below.

In [ ]:
import sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/NaifMersal/time-series-analysis-and-forecasting.git /content/ts-course
    %cd /content/ts-course
    !pip install -q -e .

from coursekit import datasets as D
from coursekit import plotting as P

P.use_course_style()
spine = D.spine()
print(f"Loaded {len(spine)} months, "
      f"{spine['ds'].min():%b %Y} to {spine['ds'].max():%b %Y}")
P.plot_series(spine, title="If you can see this chart, you are ready.")

---
## 2. Dates in pandas

A time series is a value **plus a timestamp**, and almost every real-world bug in
forecasting is a timestamp bug. Three things to be fluent in.

In [ ]:
import numpy as np
import pandas as pd

# A DatetimeIndex is not a list of strings.
dates = pd.date_range("2024-01-01", periods=6, freq="MS")   # MS = month START
print(dates)
print("\nyear:", dates.year.tolist())
print("month:", dates.month.tolist())

`freq="MS"` is month-start, `"ME"` is month-end, `"D"` daily, `"h"` hourly,
`"QS"` quarter-start. Getting this wrong shifts your whole series by a month.

In [ ]:
# resample: change the frequency. asfreq: assert one, exposing gaps as NaN.
daily = pd.Series(np.arange(60.0), index=pd.date_range("2024-01-01", periods=60, freq="D"))
print("daily -> monthly totals:")
print(daily.resample("MS").sum())

# Now punch a hole in the calendar and make it visible.
holed = daily.drop(daily.index[10:14])
print(f"\nrows after dropping 4 days: {len(holed)}")
print(f"rows after asfreq('D'):     {len(holed.asfreq('D'))}  "
      f"<- the gap is now VISIBLE as NaN")

**Why this matters.** Dropping four rows does not shift the index - but it *does* mean
row `t-7` is no longer "one week ago". Every seasonal lag after the gap is wrong, and
nothing raises an error. Day 1 comes back to this at the end of the first hour.

In [ ]:
# TODO: `s` below is missing two months. Reindex it onto a complete monthly
#       calendar so the gaps become visible NaNs, then count them.
s = pd.Series(
    [10.0, 11, 13, 12, 15, 16],
    index=pd.to_datetime(["2024-01-01", "2024-02-01", "2024-05-01",
                          "2024-06-01", "2024-07-01", "2024-08-01"]),
)

repaired = ...

assert repaired is not Ellipsis, "Fill in `repaired` above."
print(repaired)
print(f"missing months: {int(repaired.isna().sum())}")

---
## 3. Long format, and groupby

The whole course uses the layout fpppy uses: one row per series per timestamp, with
columns **`unique_id` / `ds` / `y`**. It looks redundant for one series and pays off the
moment you have 148.

In [ ]:
allr = D.retail_all()
print(allr.head())
print(f"\n{allr['unique_id'].nunique()} series, {len(allr):,} rows")

# One number per series - the pattern used constantly on Day 1.
summary = (allr.groupby("unique_id")["y"]
               .agg(n="size", mean="mean", last="last")
               .sort_values("mean", ascending=False))
print("\nBiggest five by average turnover:")
print(summary.head())

---
## 4. A ten-minute regression refresher

You do not need to *derive* anything in this course, but two ideas from ordinary least
squares come back on Day 2.

In [ ]:
rng = np.random.default_rng(0)
x = np.linspace(0, 10, 60)
y = 3 + 1.8 * x + rng.normal(scale=2.0, size=60)

# Fit a straight line and look at what is left over.
slope, intercept = np.polyfit(x, y, 1)
fitted = intercept + slope * x
residuals = y - fitted

print(f"true slope 1.80,  estimated {slope:.3f}")
print(f"residual mean {residuals.mean():.3f} (should be ~0)")
print(f"residual sd   {residuals.std():.3f} (we generated with 2.0)")

Two things to carry into Day 2:

- **A residual is what the model failed to explain.** `residual = actual - fitted`. If
  the residuals still contain a pattern, the model is not finished.
- **Least squares assumes the residuals are independent.** Time series residuals usually
  are *not* - this month's error looks like last month's. That single fact is why
  forecasting needs its own toolkit, and it is the thread running through Day 2.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].scatter(x, y, s=18, color=P.BLUE)
axes[0].plot(x, fitted, color=P.ORANGE, lw=2)
axes[0].set_title("fit")
axes[1].scatter(x, residuals, s=18, color=P.GREY)
axes[1].axhline(0, color=P.ORANGE, lw=1.5)
axes[1].set_title("residuals - no pattern left here")
plt.show()

---
## 5. The statistics refresher

Four short checks, one per idea the refresher deck builds. Run them in order;
each `checks.check_ex_0_*` call tells you whether it worked. These are the
ideas Day 1 and Day 2 assume, so getting them right here saves time later.

In [ ]:
# Exercise 0.1 - An 80% interval is a pair of cuts
#
# A forecast interval is not a number; it is a position in a sorted list.
# Compute the 10th and 90th quantiles of the sample below, then check them.
from coursekit import checks

rng = np.random.default_rng(3)
sample = rng.normal(size=200)

lo = ...   # the 10th quantile of `sample`
hi = ...   # the 90th quantile of `sample`

checks.check_ex_0_1(sample, lo, hi)

In [ ]:
# Exercise 0.2 - Independence: the noise is independent, the residuals are not
#
# White noise has no lag-12 autocorrelation. The naive's residuals still know
# what month it is, so theirs is large. Compute both and check.
from statsforecast import StatsForecast
from statsforecast.models import Naive

H = 24
train, _ = D.train_test(spine, h=H)
sf = StatsForecast(models=[Naive()], freq=D.FREQ, n_jobs=1)
sf.forecast(df=train, h=H, fitted=True)
fv = sf.forecast_fitted_values()
resid = (fv["y"] - fv["Naive"]).dropna()

noise = D.white_noise(n=400, seed=11)["y"]

r_noise = ...    # lag-12 autocorrelation of `noise`
r_resid = ...    # lag-12 autocorrelation of `resid`

checks.check_ex_0_2(r_noise, r_resid)

In [ ]:
# Exercise 0.3 - The floor is beatable
#
# Run the Ljung-Box test on the seasonal naive's residuals at 24 lags. The
# p-value is so small it underflows - that is the opening for Day 3.
from statsmodels.stats.diagnostic import acorr_ljungbox

lb = acorr_ljungbox(resid, lags=[24], return_df=True)
lb_pvalue = ...   # the Ljung-Box p-value from `lb`

checks.check_ex_0_3(resid, lb_pvalue)

In [ ]:
# Exercise 0.4 - Detectable is not large
#
# The 1.96/sqrt(T) bound shrinks as T grows. Compute it at T=417 and T=40.
# A tiny effect at the large T clears a tight bound; a large effect at the
# small T sits inside a loose one.
bound_large_T = ...   # 1.96 / sqrt(417)
bound_small_T = ...   # 1.96 / sqrt(40)

checks.check_ex_0_4(bound_large_T, bound_small_T)

---
## You are ready

If the chart in section 1 rendered and `check_env.py` printed "Ready", you are set.

Bring: a laptop that can run the cells above, and one time series from your own work if
you have one - the last exercise on Day 1 is easy to point at your own data.